# Python Dictionaries

> 📘 **Python Mastery** · Module 04 — Data Structures · Lesson 4/5

A dictionary maps **keys to values** — like a real dictionary maps words to definitions. Instead of remembering that item 3 is the email, you simply ask for `user["email"]`. Dicts are everywhere: JSON from an API, model configuration, a tokenizer's vocabulary — all dictionaries.

## 🎯 Learning Objectives

By the end of this lesson you will be able to:

- **Create** dictionaries with literals and read values with `[]` vs `.get()`.
- **Add, change, and remove** entries using direct assignment, `pop`, `popitem`, `del`, and `clear`.
- **Use the three views** — `keys()`, `values()`, `items()` — and explain why they are live, not copies.
- **Loop** over a dictionary correctly and test membership with `in`.
- **Merge and default** with `update()` and `setdefault()`.
- **Build nested dictionaries** (a mini user database) and state the rule that keys must be immutable/hashable.

## 1. Creating a Dictionary

Each entry pairs a key with a value: `"name": "Sarah"`. Keys must be unique within a dict; values can be anything and may repeat. The constructor forms are handy when the data starts life as pairs or keyword arguments.

**Syntax:**
```python
my_dict = {"key1": value1, "key2": value2}
empty = {}
from_pairs = dict([("a", 1), ("b", 2)])
from_kwargs = dict(a=1, b=2)
```

In [1]:
student = {"name": "Sarah", "age": 22, "city": "Dhaka"}
empty = {}
from_pairs = dict([("a", 1), ("b", 2)])
from_kwargs = dict(x=10, y=20)

print(student)
print(student["name"], "-", student["age"])
print(from_pairs, "|", from_kwargs)
print(len(student), "entries")

{'name': 'Sarah', 'age': 22, 'city': 'Dhaka'}
Sarah - 22
{'a': 1, 'b': 2} | {'x': 10, 'y': 20}
3 entries


## 2. Reading Values — `[key]` vs `.get()`

Two ways in, two different personalities. `d[key]` crashes with `KeyError` on a missing key — good when absence means a bug. `d.get(key)` returns `None` (or your chosen default) instead of crashing — good when absence is normal business.

**Syntax:**
```python
d[key]                 # KeyError if missing
d.get(key)             # None if missing
d.get(key, default)    # your own fallback value
```

In [2]:
stock = {"apple": 12, "banana": 7}

print(stock["apple"])            # direct access

try:
    print(stock["mango"])        # missing key -> crash
except KeyError as e:
    print("KeyError:", e)

print(stock.get("mango"))        # None instead of an exception
mango_count = stock.get("mango", 0)   # ...or a sensible default
print("mangoes in stock:", mango_count)

12
KeyError: 'mango'
None
mangoes in stock: 0


## 3. Adding and Changing Entries

One syntax covers both jobs: assign to a key. New key → entry added; existing key → value overwritten. There is no separate "append" for dicts because position does not matter — only the key does.

**Syntax:**
```python
d[new_key] = value       # add
d[existing_key] = value  # overwrite
```

In [3]:
profile = {"user": "sarah_dev"}

profile["email"] = "sarah@mail.com"    # brand-new key -> added
profile["user"] = "sarah2024"          # existing key -> replaced
print(profile)

profile["level"] = 5
print(profile)
print(len(profile), "keys now")

{'user': 'sarah2024', 'email': 'sarah@mail.com'}
{'user': 'sarah2024', 'email': 'sarah@mail.com', 'level': 5}
3 keys now


## 4. Removing Entries — `pop()`, `popitem()`, `del`, `clear()`

Four exits: `pop(key)` removes a key **and returns its value**, `popitem()` removes the most recently inserted pair as a tuple, `del d[key]` deletes without returning, and `clear()` wipes everything.

**Syntax:**
```python
v = d.pop(key)        # remove key, return its value
pair = d.popitem()    # remove + return last-inserted (key, value)
del d[key]            # delete by key, no return
d.clear()             # empty the dict
```

In [4]:
cache = {"img1": "blob-a", "img2": "blob-b", "tmp": "junk-data"}

evicted = cache.pop("tmp")         # get the value out while removing it
print(evicted, "|", cache)

last_in = cache.popitem()          # last pair inserted leaves first
print(last_in, "|", cache)

del cache["img1"]                  # delete by key, returns nothing
print(cache)

cache.clear()
print(cache)

try:
    cache.pop("missing")           # pop without default -> KeyError
except KeyError as e:
    print("KeyError:", e)

junk-data | {'img1': 'blob-a', 'img2': 'blob-b'}
('img2', 'blob-b') | {'img1': 'blob-a'}
{}
{}
KeyError: 'missing'


## 5. The Three Views — `keys()`, `values()`, `items()`

Each dict exposes three *views* onto its contents: just the keys, just the values, or `(key, value)` tuples. They behave like collections you can loop over, convert with `list(...)`, or size with `len()`.

> 🔍 **Under the Hood:** views are **live windows**, not snapshots. They hold no data themselves — they read straight from the dict at the moment you look. Add or delete an entry after creating a view and the view reflects it instantly. That saves memory (no duplicate storage) but surprises anyone expecting a frozen copy.

**Syntax:**
```python
d.keys()      # view of keys
d.values()    # view of values
d.items()     # view of (key, value) tuples
```

In [5]:
scores = {"math": 90, "science": 85}

k = scores.keys()
v = scores.values()
i = scores.items()

scores["history"] = 74            # modify AFTER creating all three views...

print(list(k))                    # ...they see it: views are LIVE
print(list(v))
print(list(i))

snapshot = list(scores.items())   # wrap in list() if you need a frozen copy
scores["art"] = 60
print(snapshot, "<- snapshot stayed put")

['math', 'science', 'history']
[90, 85, 74]
[('math', 90), ('science', 85), ('history', 74)]
[('math', 90), ('science', 85), ('history', 74)] <- snapshot stayed put


## 6. Looping Through a Dictionary

Looping over the dict itself yields keys one by one. Almost always you want both halves — use `for key, value in d.items():` and unpack them right in the loop header.

**Syntax:**
```python
for key in d:                     # keys only
for key, value in d.items():      # keys AND values (preferred)
for v in d.values():              # values only
```

In [6]:
menu = {"coffee": 3.5, "latte": 4.25, "tea": 2.75}

for drink in menu:                       # bare loop -> keys
    print(drink, end=" | ")
print()

for drink, price in menu.items():        # the everyday pattern
    print(f"{drink:<8} ${price:.2f}")

best = max(menu, key=menu.get)           # sort/rank KEYS by their VALUES
print("most expensive:", best)

coffee | latte | tea | 
coffee   $3.50
latte    $4.25
tea      $2.75
most expensive: latte


## 7. Membership — `in` Checks Keys

`x in d` asks whether `x` exists **as a key** — values are invisible to it unless you go through `d.values()`. This check is O(1) average: dicts are hash tables under the hood too.

**Syntax:**
```python
key in d               # True if key exists
value in d.values()    # explicit value search
```

In [7]:
users = {"sarah": "online", "rafi": "offline"}

print("sarah" in users)             # keys by default -> True
print("online" in users)            # False! values are NOT searched
print("online" in users.values())   # ask values explicitly

user = "mina"                       # Simulated input: login lookup
if user not in users:
    print(user, "has no account")

True
False
True
mina has no account


## 8. Merging and Defaulting — `update()` and `setdefault()`

`update()` copies pairs from another dict (or iterable of pairs) into yours, overwriting on collision — the standard merge tool. `setdefault(key, default)` reads like a sentence: "give me `d[key]`; if it doesn't exist yet, create it as `default` first." It shines for building grouped results.

**Syntax:**
```python
d.update(other_dict)                  # merge / overwrite
d.setdefault(key, default_value)      # fetch-or-create
```

In [8]:
prefs = {"theme": "dark"}

prefs.update({"theme": "light", "lang": "bn"})    # overwrite + add together
print(prefs)

log = {}
log.setdefault("monday", []).append(9)    # "monday" created with []
log.setdefault("monday", []).append(10)   # now found -> same list reused
log.setdefault("tuesday", []).append(8)
print(log)

{'theme': 'light', 'lang': 'bn'}
{'monday': [9, 10], 'tuesday': [8]}


## 9. Nested Dictionaries — a Mini User Database

Values may be dicts themselves, giving you tree-shaped records exactly like JSON. Chain brackets to descend: `db["u001"]["name"]`. This shape — IDs mapping to record dicts containing lists — appears constantly in web backends and data pipelines.

**Syntax:**
```python
db = {
    "id1": {"field": value, ...},
    "id2": {"field": value, ...},
}
db["id1"]["field"]
```

In [9]:
users_db = {
    "u001": {"name": "Sarah", "age": 22, "premium": True,
             "orders": ["book", "pen"]},
    "u002": {"name": "Rafi", "age": 19, "premium": False,
             "orders": []},
}

print(users_db["u001"]["name"])
users_db["u002"]["orders"].append("lamp")     # reach deep and mutate
print(users_db["u002"]["orders"])

users_db["u003"] = {"name": "Mina", "age": 31,
                    "premium": True, "orders": []}   # new record

for uid, info in users_db.items():
    tier = "PREMIUM" if info["premium"] else "standard"
    print(f"{uid}: {info['name']:<6} {tier:<9} {len(info['orders'])} order(s)")

Sarah
['lamp']
u001: Sarah  PREMIUM   2 order(s)
u002: Rafi   standard  1 order(s)
u003: Mina   PREMIUM   0 order(s)


## 10. Insertion Order Is Guaranteed (Python 3.7+)

Since Python 3.7 the language promises that a dict remembers the order keys were first inserted — so looping gives you entries back in insertion sequence. (Before 3.7 this was a CPython detail; today you may rely on it everywhere.)

**Syntax:**
```python
d[first_key] = v1
d[second_key] = v2
list(d) == [first_key, second_key]     # guaranteed order
```

In [10]:
config = {}
config["model"] = "cnn-small"
config["epochs"] = 10
config["learning_rate"] = 0.001

for key, value in config.items():     # prints in insertion order, every time
    print(f"{key:<14} = {value}")
print(list(config))

model          = cnn-small
epochs         = 10
learning_rate  = 0.001
['model', 'epochs', 'learning_rate']


## 11. Keys Must Be Immutable (Hashable)

Keys work by hashing, so they must be immutable types: strings, numbers, tuples of immutables. A list can never be a key → `TypeError`. Values carry no such restriction — lists, dicts, anything goes.

> 🔍 **Under the Hood:** the hash table stores each key's `hash()` to find its slot. If a key could mutate after insertion, its hash would change while sitting in the old slot — the dict would never find it again. Immutability keeps `hash(key)` stable for life, which is the whole contract.

**Syntax:**
```python
{"ok": 1, 42: 2, (1, 2): 3}   # str/int/tuple keys: fine
{[1, 2]: "x"}                 # TypeError: unhashable type: 'list'
```

In [11]:
ok = {"name": "str key", 42: "int key", 3.14: "float key", (1, 2): "tuple key"}
print(ok[(1, 2)])

try:
    bad = {[1, 2]: "list key"}
except TypeError as e:
    print("Error:", e)

tuple key
Error: cannot use 'list' as a dict key (unhashable type: 'list')


## 12. Dict Comprehension — a Teaser

Just as lists have comprehensions, dicts can be built in one line: `{key_expr: value_expr for item in iterable}`. Full treatment lands in the next lesson — here is the taste.

**Syntax:**
```python
{k_expr: v_expr for item in iterable}
```

In [12]:
squares = {n: n ** 2 for n in range(1, 6)}
print(squares)

names = ["sarah", "rafi", "mina"]
lengths = {name: len(name) for name in names}
print(lengths)

inverted = {v: k for k, v in squares.items()}    # flip keys <-> values
print(inverted)

{1: 1, 2: 4, 3: 9, 4: 16, 5: 25}
{'sarah': 5, 'rafi': 4, 'mina': 4}
{1: 1, 4: 2, 9: 3, 16: 4, 25: 5}


## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
| :--- | :--- | :--- |
| `d[key]` on a possibly-missing key | Crashes with `KeyError` mid-program | Use `d.get(key, default)` or check `if key in d:` |
| Assuming `"some_value" in d` checks values | `in` only searches keys | Use `value in d.values()` |
| Modifying a dict while looping it (`for k in d: del d[k]`) | `RuntimeError: dictionary changed size during iteration` | Loop over `list(d.keys())` or collect keys to delete first |
| `{"a": 1, "a": 9}` | Duplicate literal keys: the second silently wins | Keep literals clean; build programmatically when duplicates possible |
| Using a list (or dict/set) as a key | Unhashable → `TypeError` | Use a string, number, or tuple instead |
| Expecting `d.keys()` captured earlier to stay fixed | Views are live and track later changes | Snapshot with `list(d.keys())` |
| Forgetting values may share references | `default = []; d[k].append(x)` style bugs where several keys point to ONE list | Create fresh objects per key (see `setdefault(k, [])`) |

## 💡 Best Practices & Pro Tips

- Loop with `.items()`; index into `d[key]` only when you truly need one known entry.
- Read safely by default: `.get()` with an explicit fallback documents "absence is fine" better than any comment.
- Prefer `dict` over parallel lists (`names[i]` + `ages[i]`) the moment lookups-by-name appear — it kills off index-alignment bugs.
- For deeply nested defaults beyond `setdefault`, look up `collections.defaultdict` in Intermediate Python.
- 🔬 **AI-engineering relevance:** JSON responses parse straight into dicts (`json.loads`); hyperparameters live in config dicts passed as `**kwargs`; tokenizers are dicts mapping token → id; experiment trackers log metrics as `{"epoch": loss}`. Fluent dict skills are table stakes for ML engineering.

## 📌 Summary

| Method / Syntax | What it does | Example |
| :--- | :--- | :--- |
| `{"k": v}` / `dict(pairs)` | create a dict | `{"name": "Sarah"}` |
| `d[k]` | read (crashes if missing) | `user["email"]` |
| `d.get(k, default)` | read safely | `stock.get("mango", 0)` |
| `d[k] = v` | add or overwrite | `profile["level"] = 5` |
| `d.pop(k)` | remove + return value | `cache.pop("tmp")` |
| `d.popitem()` | remove last-inserted pair | `cache.popitem()` |
| `del d[k]` | delete by key | `del cache["img1"]` |
| `d.clear()` | empty the dict | `cache.clear()` |
| `d.keys()` / `d.values()` / `d.items()` | three LIVE views | `for k, v in d.items():` |
| `k in d` | membership tests keys, O(1) avg | `"sarah" in users` |
| `d.update(other)` | merge/overwrite many | `prefs.update({...})` |
| `d.setdefault(k, dv)` | fetch-or-create | `log.setdefault(day, [])` |

**Key takeaways**

- Dicts store **key→value** pairs: unique, hashable keys; arbitrary values.
- Insertion order is guaranteed since Python 3.7.
- `[]` raises on missing keys; `.get()` degrades gracefully — choose deliberately.
- Views (`keys/values/items`) are live windows, not copies.

## 🔗 Next Lesson

➡️ **Lesson 5/5 — List Comprehension** (`../05_List_Comprehension/notes.ipynb`): building lists (and dicts and sets!) in a single readable line.